## 1. Setup

In [1]:
import os
from dotenv import dotenv_values, load_dotenv
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.helpers import DataConnection
import os
import json
from langchain_community.document_loaders import WebBaseLoader
from ibm_watsonx_ai.experiment import AutoAI
import pandas as pd
import sqlite3
from IPython.display import Markdown
from langchain_ibm import WatsonxEmbeddings, WatsonxLLM
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# Enable Db2 Magic Commands Extensions for Jupyter Notebook
if not os.path.isfile('db2.ipynb'):
    os.system('wget https://raw.githubusercontent.com/IBM/db2-jupyter/master/db2.ipynb')

%run db2.ipynb

<>:1708: SyntaxWarning: invalid escape sequence '\s'
<>:2305: SyntaxWarning: invalid escape sequence '\?'
/tmp/ipykernel_93937/2299624180.py:1708: SyntaxWarning: invalid escape sequence '\s'
  firstCommand = "(?:^\s*)([a-zA-Z]+)(?:\s+.*|$)"
/tmp/ipykernel_93937/2299624180.py:2305: SyntaxWarning: invalid escape sequence '\?'
  pattern = "\?\*[0-9]+"


         Install itables if you want to enable scrolling of result sets.
Db2 Extensions Loaded. Version: 2024-09-16


## 2. Loading Patients Records from a Db2 table

In [3]:
db2creds = dotenv_values('.env')

In [4]:
%sql CONNECT CREDENTIALS db2creds

Connection successful. sample @ localhost 


In [5]:
df_patients = %sql SELECT * FROM PATIENTS

In [6]:
df_patients.head(5)

,PATIENT_ID,NAME,AGE,GENDER,CHOLESTEROL_LEVEL,BLOOD_PRESSURE,SMOKING_STATUS
0,1,Allison Hill,20,Female,165,94,Non-smoker
1,2,Noah Rhodes,43,Male,193,114,Non-smoker
2,3,Angie Henderson,38,Female,152,81,Non-smoker
3,4,Daniel Wagner,26,Male,182,118,Non-smoker
4,5,Cristian Santos,37,Male,195,114,Non-smoker


In [7]:
df_patients.columns

Index(['PATIENT_ID', 'NAME', 'AGE', 'GENDER', 'CHOLESTEROL_LEVEL',
       'BLOOD_PRESSURE', 'SMOKING_STATUS'],
      dtype='object')

In [8]:
df_patients.dtypes

PATIENT_ID                    Int32
NAME                 string[python]
AGE                           Int32
GENDER               string[python]
CHOLESTEROL_LEVEL             Int32
BLOOD_PRESSURE                Int32
SMOKING_STATUS       string[python]
dtype: object

In [9]:
feat_cols = ['AGE', 'GENDER', 'CHOLESTEROL_LEVEL', 'BLOOD_PRESSURE', 'SMOKING_STATUS']

# 3. Generating Row Embeddings Using `wx.ai` Embedding API

In [10]:
# Combine all columns into a single string for each row, including column names
df_patients['combined'] = df_patients.apply(
    lambda row: ' [SEP] '.join([f"{col_name}: {row[col_name]}" for col_name in feat_cols]), 
    axis=1
)

In [11]:
df_patients.iloc[0]['combined']

'AGE: 20 [SEP] GENDER: Female [SEP] CHOLESTEROL_LEVEL: 165 [SEP] BLOOD_PRESSURE: 94 [SEP] SMOKING_STATUS: Non-smoker'

In [12]:
df_patients.head()

,PATIENT_ID,NAME,AGE,GENDER,CHOLESTEROL_LEVEL,BLOOD_PRESSURE,SMOKING_STATUS,combined
0,1,Allison Hill,20,Female,165,94,Non-smoker,AGE: 20 [SEP] GENDER: Female [SEP] CHOLESTEROL...
1,2,Noah Rhodes,43,Male,193,114,Non-smoker,AGE: 43 [SEP] GENDER: Male [SEP] CHOLESTEROL_L...
2,3,Angie Henderson,38,Female,152,81,Non-smoker,AGE: 38 [SEP] GENDER: Female [SEP] CHOLESTEROL...
3,4,Daniel Wagner,26,Male,182,118,Non-smoker,AGE: 26 [SEP] GENDER: Male [SEP] CHOLESTEROL_L...
4,5,Cristian Santos,37,Male,195,114,Non-smoker,AGE: 37 [SEP] GENDER: Male [SEP] CHOLESTEROL_L...


In [13]:
load_dotenv(os.getcwd()+"/.env", override=True)

True

In [14]:
credentials = Credentials(
                url = "https://us-south.ml.cloud.ibm.com",
                api_key = os.getenv("WATSONX_APIKEY", "")
                )

client = APIClient(credentials)

project_id = os.getenv("WATSONX_PROJECT", "")
client.set.default_project(project_id)

'SUCCESS'

Set up embedding model reference

In [15]:
embeddings = Embeddings(
    model_id=client.foundation_models.EmbeddingModels.MULTILINGUAL_E5_LARGE,
    credentials=credentials,
    project_id=project_id,
)

In [16]:
row_combined = df_patients['combined'].tolist()

In [17]:
len(row_combined)

20

In [18]:
row_combined[0]

'AGE: 20 [SEP] GENDER: Female [SEP] CHOLESTEROL_LEVEL: 165 [SEP] BLOOD_PRESSURE: 94 [SEP] SMOKING_STATUS: Non-smoker'

In [19]:
patient_vectors = embeddings.embed_documents(texts=row_combined)

In [20]:
df_patients['embedding'] = patient_vectors
df_patients['embedding'] = df_patients['embedding'].apply(lambda x: '[' + ', '.join(map(str, x)) + ']')

In [21]:
df_patients.drop(['combined'], axis=1, inplace=True)

In [22]:
df_patients.head()

,PATIENT_ID,NAME,AGE,GENDER,CHOLESTEROL_LEVEL,BLOOD_PRESSURE,SMOKING_STATUS,embedding
0,1,Allison Hill,20,Female,165,94,Non-smoker,"[0.016166046, -0.024523495, -0.05954662, -0.04..."
1,2,Noah Rhodes,43,Male,193,114,Non-smoker,"[0.026283592, -0.020796603, -0.041334175, -0.0..."
2,3,Angie Henderson,38,Female,152,81,Non-smoker,"[0.023451703, -0.011626242, -0.044654254, -0.0..."
3,4,Daniel Wagner,26,Male,182,118,Non-smoker,"[0.02566346, -0.009682621, -0.04426426, -0.047..."
4,5,Cristian Santos,37,Male,195,114,Non-smoker,"[0.033487085, -0.013782806, -0.045675095, -0.0..."


In [23]:
df_patients.dtypes

PATIENT_ID                    Int32
NAME                 string[python]
AGE                           Int32
GENDER               string[python]
CHOLESTEROL_LEVEL             Int32
BLOOD_PRESSURE                Int32
SMOKING_STATUS       string[python]
embedding                    object
dtype: object

In [24]:
import csv

In [25]:
# Save DataFrame to CSV with all fields quoted
df_patients.to_csv(
    'patients.csv',
    index=False,
    quoting=csv.QUOTE_NONNUMERIC
)

In [26]:
%sql DROP TABLE PATIENTS

Command completed.


In [27]:
%%sql
CREATE TABLE PATIENTS  (
		  PATIENT_ID INTEGER , 
		  NAME VARCHAR(30 OCTETS) , 
		  AGE INTEGER , 
		  GENDER VARCHAR(6 OCTETS) , 
		  CHOLESTEROL_LEVEL INTEGER , 
		  BLOOD_PRESSURE INTEGER , 
		  SMOKING_STATUS VARCHAR(10 OCTETS) , 
		  EMBEDDING VECTOR(1024,FLOAT32) ) 

Command completed.


In [28]:
%%capture output
sql = f'''"IMPORT FROM 'patients.csv' OF DEL skipcount 1 INSERT INTO PATIENTS"'''
_ = ! db2 "connect to SAMPLE"

output = %system db2 {sql}
print(output)

## 5. In-database Vector Operations at Db2

### Function 1: VECTOR_SERIALIZE - Unpacking Vectors

In [29]:
%sql SELECT NAME, VECTOR_SERIALIZE(EMBEDDING) as VECTOR FROM PATIENTS FETCH FIRST 1 ROWS ONLY

,NAME,VECTOR
0,Allison Hill,"[0.0161660463,-0.0245234948,-0.0595466197,-0.0..."


### Function 2: VECTOR_DIMENSION_COUNT

In [30]:
%sql SELECT NAME, VECTOR_DIMENSION_COUNT(EMBEDDING) FROM PATIENTS FETCH FIRST 3 ROWS ONLY

,NAME,2
0,Allison Hill,1024
1,Noah Rhodes,1024
2,Angie Henderson,1024


### FUNCTION 3: VECTOR_NORM

In [31]:
%%sql 
SELECT NAME, VECTOR_NORM(EMBEDDING, EUCLIDEAN) as NORM 
FROM PATIENTS 
WHERE PATIENT_ID = 2

,NAME,NORM
0,Noah Rhodes,1.0


### Function 4: VECTOR_DISTANCE
* Input: vector1, vector2
* Output: distance

Query patient: PATIENT_ID = 2

In [32]:
%%sql
SELECT NAME, AGE, GENDER, CHOLESTEROL_LEVEL, SMOKING_STATUS
FROM PATIENTS
WHERE PATIENT_ID = 2

,NAME,AGE,GENDER,CHOLESTEROL_LEVEL,SMOKING_STATUS
0,Noah Rhodes,43,Male,193,Non-smoker


In [33]:
%%sql 
SELECT NAME, AGE, GENDER, CHOLESTEROL_LEVEL, SMOKING_STATUS, VECTOR_DISTANCE((SELECT EMBEDDING FROM PATIENTS WHERE PATIENT_ID = 2), EMBEDDING, EUCLIDEAN) as DISTANCE
FROM PATIENTS
WHERE PATIENT_ID <> 2
ORDER BY DISTANCE ASC
FETCH FIRST 3 ROWS ONLY

,NAME,AGE,GENDER,CHOLESTEROL_LEVEL,SMOKING_STATUS,DISTANCE
0,Abe Shaffer,44,Male,194,Non-smoker,0.130702
1,Michele Williams,47,Male,156,Non-smoker,0.188103
2,Cristian Santos,37,Male,195,Non-smoker,0.217936
